# Deployment of Llama 3.3 70B Instruct for Gov/Non-Gov Org Classification

This notebook applies the best-performing few-shot Llama 3.3 70B Instruct model to unique representative organization names.

- Class 1: Government-related funding organization
- Class 0: Non-government-related funding organization
- Deployment dataset: Unique `Representative_Organization_Name` values
- Few-shot examples: 10 balanced examples from the labeled BC dataset
- Final output: `Representative_Organization_Name`, `Representative_Instance_id`, and `Class`

In [10]:
import boto3

sm_client = boto3.client(
    "sagemaker",
    region_name="us-east-2"
)

response = sm_client.list_endpoints(
    SortBy="CreationTime",
    SortOrder="Descending"
)

for endpoint in response["Endpoints"]:
    print(
        endpoint["EndpointName"],
        endpoint["EndpointStatus"]
    )

llama3-3-70b-gov-nongov-deployment-bc Failed


In [11]:
OLD_ENDPOINT_NAME = "llama3-3-70b-gov-nongov-deployment-bc"

sm_client.delete_endpoint(
    EndpointName=OLD_ENDPOINT_NAME
)

print(
    f"Deletion requested for: "
    f"{OLD_ENDPOINT_NAME}"
)

Deletion requested for: llama3-3-70b-gov-nongov-deployment-bc


In [12]:
import time

while True:
    try:
        response = sm_client.describe_endpoint(
            EndpointName=OLD_ENDPOINT_NAME
        )

        print(
            "Current status:",
            response["EndpointStatus"]
        )

        time.sleep(15)

    except sm_client.exceptions.ClientError:
        print(
            "Endpoint has been completely deleted."
        )
        break

Endpoint has been completely deleted.


In [13]:
import boto3

REGION = "us-east-2"
RESOURCE_NAME = "llama3-3-70b-gov-nongov-deployment-bc"

sm_client = boto3.client(
    "sagemaker",
    region_name=REGION
)

# =============================================================================
# 1. Delete the endpoint configuration
# =============================================================================

try:
    sm_client.describe_endpoint_config(
        EndpointConfigName=RESOURCE_NAME
    )

    sm_client.delete_endpoint_config(
        EndpointConfigName=RESOURCE_NAME
    )

    print(
        f"Deleted endpoint configuration: "
        f"{RESOURCE_NAME}"
    )

except sm_client.exceptions.ClientError as error:
    print(
        "Endpoint configuration does not exist "
        "or could not be deleted."
    )
    print(error)

# =============================================================================
# 2. Delete the SageMaker model if it exists under the same name
# =============================================================================

try:
    sm_client.describe_model(
        ModelName=RESOURCE_NAME
    )

    sm_client.delete_model(
        ModelName=RESOURCE_NAME
    )

    print(
        f"Deleted SageMaker model: "
        f"{RESOURCE_NAME}"
    )

except sm_client.exceptions.ClientError:
    print(
        "No SageMaker model exists under the same name."
    )

Deleted endpoint configuration: llama3-3-70b-gov-nongov-deployment-bc
No SageMaker model exists under the same name.


In [14]:
try:
    sm_client.describe_endpoint_config(
        EndpointConfigName=RESOURCE_NAME
    )

    print(
        "Endpoint configuration still exists."
    )

except sm_client.exceptions.ClientError:
    print(
        "Endpoint configuration has been completely deleted."
    )

Endpoint configuration has been completely deleted.


In [15]:
# Install only the packages required for SageMaker inference and data processing.
%pip install -q \
    "sagemaker<3" \
    scikit-learn \
    pandas \
    numpy \
    openpyxl

Note: you may need to restart the kernel to use updated packages.


## 1. Import Libraries

In [1]:
import json
import random
import re
import time
from pathlib import Path
from typing import Any

import boto3
import numpy as np
import pandas as pd
import sagemaker

from sagemaker.jumpstart.model import JumpStartModel

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


## 2. Experiment Configuration

In [2]:
# =============================================================================
# File settings
# =============================================================================

DEPLOYMENT_DATA_FILE = Path(
    "./deployment_final_ond_clusters.csv"
)

LABELED_DATA_FILE = Path(
    "./org_name_BC.csv"
)

FINAL_OUTPUT_FILE = Path(
    "./deployment_llama3_70b_BC_classification.csv"
)

AUDIT_OUTPUT_FILE = Path(
    "./deployment_llama3_70b_BC_classification_audit.csv"
)

CHECKPOINT_FILE = Path(
    "./deployment_llama3_70b_checkpoint.csv"
)

FEW_SHOT_EXAMPLES_FILE = Path(
    "./deployment_few_shot_examples.csv"
)

# =============================================================================
# Column settings
# =============================================================================

DEPLOYMENT_TEXT_COLUMN = "Representative_Organization_Name"
DEPLOYMENT_ID_COLUMN = "Representative_Instance_id"

LABELED_TEXT_COLUMN = "Org_Name"
LABELED_CLASS_COLUMN = "Class"

# =============================================================================
# Model settings
# =============================================================================

MODEL_ID = "meta-textgeneration-llama-3-3-70b-instruct"
MODEL_VERSION = "*"

ENDPOINT_INSTANCE_TYPE = "ml.g5.48xlarge"

ENDPOINT_NAME = "llama3-3-70b-gov-nongov-deployment-bc"

ACCEPT_EULA = True

# =============================================================================
# Few-shot settings
# =============================================================================

RANDOM_SEED = 42

N_FEW_SHOT_PER_CLASS = 5
N_FEW_SHOT_EXAMPLES = 10

# =============================================================================
# Inference settings
# =============================================================================

MAX_NEW_TOKENS = 4

TEMPERATURE = 0.0
DO_SAMPLE = False

MAX_RETRIES = 5
INITIAL_RETRY_DELAY = 3

# Save progress after this many completed predictions.
CHECKPOINT_INTERVAL = 25

# Optional delay between requests.
REQUEST_DELAY_SECONDS = 0.0

print(f"Model: {MODEL_ID}")
print(f"Endpoint instance: {ENDPOINT_INSTANCE_TYPE}")

Model: meta-textgeneration-llama-3-3-70b-instruct
Endpoint instance: ml.g5.48xlarge


## 3. Load and Prepare the Deployment Dataset

In [3]:
# =============================================================================
# Load the deployment dataset using UTF-8 encoding
# =============================================================================

deployment_df = pd.read_csv(
    DEPLOYMENT_DATA_FILE,
    encoding="utf-8-sig"
)

required_columns = {
    DEPLOYMENT_TEXT_COLUMN,
    DEPLOYMENT_ID_COLUMN
}

missing_columns = required_columns.difference(
    deployment_df.columns
)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_columns)}"
    )

print(
    f"Original number of rows: "
    f"{len(deployment_df):,}"
)

Original number of rows: 61,481


In [4]:
# =============================================================================
# Keep unique representative organization name and ID pairs
# =============================================================================

unique_org_df = (
    deployment_df[
        [
            DEPLOYMENT_TEXT_COLUMN,
            DEPLOYMENT_ID_COLUMN
        ]
    ]
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
)

# Remove leading and trailing whitespace without altering Unicode characters.
unique_org_df[DEPLOYMENT_TEXT_COLUMN] = (
    unique_org_df[DEPLOYMENT_TEXT_COLUMN]
    .astype(str)
    .str.strip()
)

# =============================================================================
# Validate the one-to-one relationship
# =============================================================================

names_with_multiple_ids = (
    unique_org_df
    .groupby(DEPLOYMENT_TEXT_COLUMN)[DEPLOYMENT_ID_COLUMN]
    .nunique()
    .gt(1)
    .sum()
)

ids_with_multiple_names = (
    unique_org_df
    .groupby(DEPLOYMENT_ID_COLUMN)[DEPLOYMENT_TEXT_COLUMN]
    .nunique()
    .gt(1)
    .sum()
)

if names_with_multiple_ids > 0:
    raise ValueError(
        "At least one representative organization name "
        "maps to multiple representative IDs."
    )

if ids_with_multiple_names > 0:
    raise ValueError(
        "At least one representative ID "
        "maps to multiple organization names."
    )

print(
    f"Unique representative organizations: "
    f"{len(unique_org_df):,}"
)

print(
    f"Unique representative IDs: "
    f"{unique_org_df[DEPLOYMENT_ID_COLUMN].nunique():,}"
)

display(unique_org_df.head())

Unique representative organizations: 36,618
Unique representative IDs: 36,618


,Representative_Organization_Name,Representative_Instance_id
0,National Science Foundation,1000001
1,National Natural Science Foundation of China,1000002
2,Conselho Nacional de Desenvolvimento Cientific...,1000003
3,Natural Environment Research Council,1000004
4,Natural Sciences and Engineering Research Coun...,1000005


## 4. Prepare Fixed Few-Shot Examples

In [5]:
# =============================================================================
# Create or reload a fixed set of few-shot examples
# =============================================================================

if FEW_SHOT_EXAMPLES_FILE.exists():

    few_shot_examples = pd.read_csv(
        FEW_SHOT_EXAMPLES_FILE,
        encoding="utf-8-sig"
    )

    print(
        "Loaded previously saved few-shot examples."
    )

else:

    labeled_df = pd.read_csv(
        LABELED_DATA_FILE,
        encoding="utf-8-sig"
    )

    required_labeled_columns = {
        LABELED_TEXT_COLUMN,
        LABELED_CLASS_COLUMN
    }

    missing_labeled_columns = (
        required_labeled_columns.difference(
            labeled_df.columns
        )
    )

    if missing_labeled_columns:
        raise ValueError(
            f"Missing labeled-data columns: "
            f"{sorted(missing_labeled_columns)}"
        )

    labeled_df = labeled_df.dropna(
        subset=[
            LABELED_TEXT_COLUMN,
            LABELED_CLASS_COLUMN
        ]
    ).copy()

    labeled_df[LABELED_CLASS_COLUMN] = (
        labeled_df[LABELED_CLASS_COLUMN]
        .astype(int)
    )

    class_0_examples = (
        labeled_df[
            labeled_df[LABELED_CLASS_COLUMN] == 0
        ]
        .sample(
            n=N_FEW_SHOT_PER_CLASS,
            random_state=RANDOM_SEED
        )
    )

    class_1_examples = (
        labeled_df[
            labeled_df[LABELED_CLASS_COLUMN] == 1
        ]
        .sample(
            n=N_FEW_SHOT_PER_CLASS,
            random_state=RANDOM_SEED
        )
    )

    few_shot_examples = pd.concat(
        [
            class_0_examples,
            class_1_examples
        ],
        ignore_index=True
    )

    few_shot_examples = (
        few_shot_examples
        .sample(
            frac=1,
            random_state=RANDOM_SEED
        )
        .reset_index(drop=True)
    )

    few_shot_examples[
        [
            LABELED_TEXT_COLUMN,
            LABELED_CLASS_COLUMN
        ]
    ].to_csv(
        FEW_SHOT_EXAMPLES_FILE,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        "Created and saved fixed few-shot examples."
    )

display(
    few_shot_examples[
        [
            LABELED_TEXT_COLUMN,
            LABELED_CLASS_COLUMN
        ]
    ]
)

Loaded previously saved few-shot examples.


,Org_Name,Class
0,Direccion General de Proteccion de la Naturaleza,1
1,Park Cities Quail,0
2,Welsh Government's European Social Fund,1
3,Flora of North America,0
4,Long-Term Ecological Research program,1
5,Societas Entomologica Fennica,0
6,National Basic Research Program of China,1
7,Questar Exploration and Production,0
8,Northamptonshire Butterfly Conservation,0
9,State Forestry Bureau of China,1


## 5. Define the Classification Prompt

In [6]:
SYSTEM_INSTRUCTION = """
You are a binary classifier of funding organization names.

Classification rule:

Class 1:
The organization is a government organization, government agency,
government ministry, government department, national or subnational
public authority, public research council, or a funding organization
that is directly created, controlled, or operated by a government.

Class 0:
The organization is not government-related. This includes private
foundations, charities, companies, associations, universities,
nonprofit organizations, professional societies, and other
non-government organizations.

Decision rule:
Assign Class 1 only when the organization is clearly government-related.
If the organization does not satisfy the Class 1 definition, assign
Class 0.

Return exactly one character:
0 or 1

Do not provide explanations, punctuation, words, or additional text.
""".strip()

In [7]:
# =============================================================================
# Llama chat format
# =============================================================================


def format_llama3_chat_prompt(
    system_instruction: str,
    user_content: str
) -> str:
    """Format the prompt using the Llama 3 chat template."""

    return (
        "<|begin_of_text|>"
        "<|start_header_id|>system<|end_header_id|>\n\n"
        f"{system_instruction}"
        "<|eot_id|>"
        "<|start_header_id|>user<|end_header_id|>\n\n"
        f"{user_content}"
        "<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n\n"
    )

In [8]:
# =============================================================================
# Few-shot prompt
# =============================================================================


def build_few_shot_prompt(
    organization_name: str
) -> str:
    """Build the fixed few-shot classification prompt."""

    example_lines = []

    for example_number, (_, row) in enumerate(
        few_shot_examples.iterrows(),
        start=1
    ):

        example_lines.append(
            f"Example {example_number}\n"
            f"Organization name: "
            f"{row[LABELED_TEXT_COLUMN]}\n"
            f"Class: "
            f"{int(row[LABELED_CLASS_COLUMN])}"
        )

    examples_text = "\n\n".join(
        example_lines
    )

    user_content = (
        "Use the labeled examples below as guidance.\n\n"
        f"{examples_text}\n\n"
        "Now classify the following organization.\n\n"
        f"Organization name: {organization_name}\n\n"
        "Answer:"
    )

    return format_llama3_chat_prompt(
        system_instruction=SYSTEM_INSTRUCTION,
        user_content=user_content
    )

## 6. Deploy Llama 3.3 70B Instruct

In [9]:
# =============================================================================
# Deploy the SageMaker JumpStart endpoint
# =============================================================================

jumpstart_model = JumpStartModel(
    model_id=MODEL_ID,
    model_version=MODEL_VERSION,
    instance_type=ENDPOINT_INSTANCE_TYPE,
    env={
        "OPTION_GPU_MEMORY_UTILIZATION": "0.90",
        "OPTION_MAX_MODEL_LEN": "8192",
        "OPTION_TENSOR_PARALLEL_DEGREE": "8",
        "OPTION_MAX_ROLLING_BATCH_SIZE": "4"
    }
)

predictor = jumpstart_model.deploy(
    accept_eula=ACCEPT_EULA,
    endpoint_name=ENDPOINT_NAME,
    instance_type=ENDPOINT_INSTANCE_TYPE
)

print(
    f"Endpoint deployed successfully: "
    f"{predictor.endpoint_name}"
)

Using model 'meta-textgeneration-llama-3-3-70b-instruct' with wildcard version identifier '*'. You can pin to version '1.53.0' for more stable results. Note that models may have different input/output signatures after a major version upgrade.
/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/model.py:347: SageMakerV2DeprecationWarning: JumpStartModel is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelBuilder` (`from sagemaker.serve import ModelBuilder`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
JumpStartModel is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelBuilder` (`from sagemaker.serve import ModelBuilder`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1

----------------!

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/base_predictor.py:140: SageMakerV2DeprecationWarning: Predictor is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `sagemaker.core.resources.Endpoint`.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
Predictor is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `sagemaker.core.resources.Endpoint`.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Endpoint deployed successfully: llama3-3-70b-gov-nongov-deployment-bc


## 7. Define Endpoint Response Parsing

In [10]:
def extract_generated_text(
    response: Any
) -> str:
    """Extract generated text from common SageMaker response formats."""

    if isinstance(response, str):
        return response

    if isinstance(response, bytes):
        return response.decode(
            "utf-8",
            errors="replace"
        )

    if isinstance(response, dict):

        for key in [
            "generated_text",
            "generation",
            "text",
            "output"
        ]:

            if key in response:
                return str(response[key])

    if isinstance(response, list) and response:

        first_item = response[0]

        if isinstance(first_item, str):
            return first_item

        if isinstance(first_item, dict):

            for key in [
                "generated_text",
                "generation",
                "text",
                "output"
            ]:

                if key in first_item:
                    return str(first_item[key])

    return str(response)

In [11]:
# =============================================================================
# Conservative parsing
# =============================================================================

def parse_binary_class(
    generated_text: str
) -> tuple[int, bool]:
    """
    Parse the binary classification output.

    Class 1 is assigned only when the model explicitly returns 1.
    All other cases are conservatively assigned to Class 0.
    """

    cleaned_text = (
        generated_text
        .replace("<|eot_id|>", "")
        .strip()
    )

    # Preferred response format.
    if cleaned_text == "1":
        return 1, True

    if cleaned_text == "0":
        return 0, True

    # Recover a standalone class label from a noncompliant response.
    match = re.search(
        r"(?<!\d)([01])(?!\d)",
        cleaned_text
    )

    if match:

        parsed_class = int(
            match.group(1)
        )

        return parsed_class, False

    # Apply the conservative decision rule.
    return 0, False

## 8. Define Endpoint Invocation with Retry Handling

In [12]:
def invoke_classifier(
    prompt: str
) -> dict:
    """Invoke the SageMaker endpoint with retry handling."""

    payload = {
        "inputs": prompt,
        "parameters": {
            "max_new_tokens": MAX_NEW_TOKENS,
            "temperature": TEMPERATURE,
            "do_sample": DO_SAMPLE,
            "return_full_text": False,
            "stop": [
                "<|eot_id|>",
                "\n"
            ]
        }
    }

    last_exception = None

    for attempt in range(
        1,
        MAX_RETRIES + 1
    ):

        try:

            response = predictor.predict(
                payload
            )

            generated_text = extract_generated_text(
                response
            )

            predicted_class, valid_format = (
                parse_binary_class(
                    generated_text
                )
            )

            if REQUEST_DELAY_SECONDS > 0:
                time.sleep(
                    REQUEST_DELAY_SECONDS
                )

            return {
                "predicted_class": predicted_class,
                "raw_response": generated_text,
                "valid_format": valid_format,
                "attempts": attempt
            }

        except Exception as exception:

            last_exception = exception

            if attempt == MAX_RETRIES:
                break

            delay = (
                INITIAL_RETRY_DELAY
                * (2 ** (attempt - 1))
            )

            print(
                f"Endpoint call failed on attempt "
                f"{attempt}. "
                f"Retrying in {delay} seconds."
            )

            time.sleep(delay)

    raise RuntimeError(
        "Endpoint invocation failed after all retries."
    ) from last_exception

## 9. Test the Endpoint Before Full Deployment

In [13]:
test_organizations = [
    "National Science Foundation",
    "Harvard University"
]

for organization_name in test_organizations:

    test_prompt = build_few_shot_prompt(
        organization_name
    )

    test_result = invoke_classifier(
        test_prompt
    )

    print(
        organization_name,
        "->",
        test_result
    )

National Science Foundation -> {'predicted_class': 1, 'raw_response': '1', 'valid_format': True, 'attempts': 1}
Harvard University -> {'predicted_class': 0, 'raw_response': '0', 'valid_format': True, 'attempts': 1}


## 10. Run Deployment with Automatic Checkpointing and Resume Support

In [14]:
# =============================================================================
# Load existing progress if available
# =============================================================================

if CHECKPOINT_FILE.exists():

    checkpoint_df = pd.read_csv(
        CHECKPOINT_FILE,
        encoding="utf-8-sig"
    )

    processed_ids = set(
        checkpoint_df[
            DEPLOYMENT_ID_COLUMN
        ].tolist()
    )

    prediction_records = (
        checkpoint_df.to_dict(
            orient="records"
        )
    )

    print(
        f"Loaded checkpoint with "
        f"{len(processed_ids):,} completed organizations."
    )

else:

    processed_ids = set()
    prediction_records = []

    print(
        "No existing checkpoint found. "
        "Starting a new deployment."
    )

No existing checkpoint found. Starting a new deployment.


In [15]:
# =============================================================================
# Main Deployment Loop
# =============================================================================

total_instances = len(
    unique_org_df
)

completed_before_start = len(
    processed_ids
)

print(
    f"Total unique organizations: "
    f"{total_instances:,}"
)

print(
    f"Already completed: "
    f"{completed_before_start:,}"
)

for row_number, (_, row) in enumerate(
    unique_org_df.iterrows(),
    start=1
):

    representative_id = row[
        DEPLOYMENT_ID_COLUMN
    ]

    organization_name = row[
        DEPLOYMENT_TEXT_COLUMN
    ]

    # Skip organizations that were already classified.
    if representative_id in processed_ids:
        continue

    prompt = build_few_shot_prompt(
        organization_name
    )

    result = invoke_classifier(
        prompt
    )

    prediction_records.append(
        {
            DEPLOYMENT_TEXT_COLUMN:
                organization_name,

            DEPLOYMENT_ID_COLUMN:
                representative_id,

            "Class":
                int(
                    result[
                        "predicted_class"
                    ]
                ),

            "Valid_Output_Format":
                bool(
                    result[
                        "valid_format"
                    ]
                ),

            "Raw_Response":
                result[
                    "raw_response"
                ],

            "Invocation_Attempts":
                int(
                    result[
                        "attempts"
                    ]
                )
        }
    )

    processed_ids.add(
        representative_id
    )

    current_completed = len(
        processed_ids
    )

    # -------------------------------------------------------------------------
    # Save a checkpoint at regular intervals
    # -------------------------------------------------------------------------

    if (
        current_completed % CHECKPOINT_INTERVAL == 0
        or current_completed == total_instances
    ):

        checkpoint_df = pd.DataFrame(
            prediction_records
        )

        checkpoint_df.to_csv(
            CHECKPOINT_FILE,
            index=False,
            encoding="utf-8-sig"
        )

        print(
            f"Completed "
            f"{current_completed:,}/"
            f"{total_instances:,} "
            f"organizations "
            f"({current_completed / total_instances:.1%})"
        )

Total unique organizations: 36,618
Already completed: 0
Completed 25/36,618 organizations (0.1%)
Completed 50/36,618 organizations (0.1%)
Completed 75/36,618 organizations (0.2%)
Completed 100/36,618 organizations (0.3%)
Completed 125/36,618 organizations (0.3%)
Completed 150/36,618 organizations (0.4%)
Completed 175/36,618 organizations (0.5%)
Completed 200/36,618 organizations (0.5%)
Completed 225/36,618 organizations (0.6%)
Completed 250/36,618 organizations (0.7%)
Completed 275/36,618 organizations (0.8%)
Completed 300/36,618 organizations (0.8%)
Completed 325/36,618 organizations (0.9%)
Completed 350/36,618 organizations (1.0%)
Completed 375/36,618 organizations (1.0%)
Completed 400/36,618 organizations (1.1%)
Completed 425/36,618 organizations (1.2%)
Completed 450/36,618 organizations (1.2%)
Completed 475/36,618 organizations (1.3%)
Completed 500/36,618 organizations (1.4%)
Completed 525/36,618 organizations (1.4%)
Completed 550/36,618 organizations (1.5%)
Completed 575/36,618 or

## 11. Validate Deployment Results

In [16]:
# =============================================================================
# Load the completed deployment results
# =============================================================================

audit_df = pd.read_csv(
    CHECKPOINT_FILE,
    encoding="utf-8-sig"
)

print(
    f"Number of predictions: "
    f"{len(audit_df):,}"
)

print(
    f"Unique representative IDs: "
    f"{audit_df[DEPLOYMENT_ID_COLUMN].nunique():,}"
)

print(
    f"Invalid output formats: "
    f"{(~audit_df['Valid_Output_Format']).sum():,}"
)

print("\nClass distribution:")

print(
    audit_df[
        "Class"
    ]
    .value_counts()
    .sort_index()
)

Number of predictions: 36,618
Unique representative IDs: 36,618
Invalid output formats: 0

Class distribution:
Class
0    20326
1    16292
Name: count, dtype: int64


In [17]:
# =============================================================================
# Validate deployment completeness
# =============================================================================

if len(audit_df) != len(unique_org_df):

    raise RuntimeError(
        f"Expected {len(unique_org_df):,} predictions, "
        f"but found {len(audit_df):,}."
    )

if audit_df[
    DEPLOYMENT_ID_COLUMN
].duplicated().any():

    raise RuntimeError(
        "Duplicate Representative_Instance_id values "
        "were detected in the deployment output."
    )

if not set(
    audit_df["Class"].unique()
).issubset({0, 1}):

    raise RuntimeError(
        "Unexpected class labels were detected."
    )

print(
    "Deployment validation completed successfully."
)

Deployment validation completed successfully.


## 12. Save the Final Classification File

In [18]:
# =============================================================================
# Create the final deployment file
# =============================================================================

final_output_df = audit_df[
    [
        DEPLOYMENT_TEXT_COLUMN,
        DEPLOYMENT_ID_COLUMN,
        "Class"
    ]
].copy()

# Preserve the representative organization order from the source data.
final_output_df = unique_org_df.merge(
    final_output_df,
    on=[
        DEPLOYMENT_TEXT_COLUMN,
        DEPLOYMENT_ID_COLUMN
    ],
    how="left",
    validate="one_to_one"
)

final_output_df.to_csv(
    FINAL_OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print(
    f"Final output saved to: "
    f"{FINAL_OUTPUT_FILE}"
)

print(
    f"Number of classified organizations: "
    f"{len(final_output_df):,}"
)

display(
    final_output_df.head(20)
)

Final output saved to: deployment_llama3_70b_BC_classification.csv
Number of classified organizations: 36,618


,Representative_Organization_Name,Representative_Instance_id,Class
0,National Science Foundation,1000001,1
1,National Natural Science Foundation of China,1000002,1
2,Conselho Nacional de Desenvolvimento Cientific...,1000003,1
3,Natural Environment Research Council,1000004,1
4,Natural Sciences and Engineering Research Coun...,1000005,1
5,Coordenacao de Aperfeicoamento de Pessoal de N...,1000006,1
6,Australian Research Council,1000007,1
7,European Commission,1000008,1
8,Fish and Wildlife Service,1000009,1
9,Consejo Nacional de Ciencia y Tecnologia,1000010,1


In [19]:
# =============================================================================
# Save the detailed audit file separately
# =============================================================================

audit_df.to_csv(
    AUDIT_OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print(
    f"Audit file saved to: "
    f"{AUDIT_OUTPUT_FILE}"
)

Audit file saved to: deployment_llama3_70b_BC_classification_audit.csv


## 13. Delete the SageMaker Endpoint

In [20]:
# =============================================================================
# Delete the endpoint after deployment is complete
# =============================================================================

predictor.delete_endpoint(
    delete_endpoint_config=True
)

print(
    f"Endpoint deleted: "
    f"{ENDPOINT_NAME}"
)

Endpoint deleted: llama3-3-70b-gov-nongov-deployment-bc


In [22]:
# =============================================================================
# Delete the associated SageMaker model object
# =============================================================================

try:

    predictor.delete_model()

    print(
        "Associated SageMaker model object deleted."
    )

except Exception as exception:

    print(
        "Endpoint deletion succeeded, "
        "but automatic model-object deletion "
        "was not completed."
    )

    print(
        f"Cleanup message: {exception}"
    )

Associated SageMaker model object deleted.
